# So sánh các thuật toán dự báo giá đồng

Notebook này chạy cùng một protocol cho bốn kiến trúc dự báo một phiên: SimpleRNN, LSTM, GRU và Bi-LSTM. Sau đó notebook so sánh riêng hai chiến lược nhiều bước dùng GRU: IMS (dự báo lặp) và DMS (dự báo trực tiếp).

Các kiến trúc một bước dùng cùng tập dữ liệu, input, cách chia, scaler và metric. IMS/DMS được đánh giá riêng theo từng lead vì đây là bài toán nhiều bước và chỉ dùng lịch sử giá đồng làm input.

Notebook paper-inspired RNN/LSTM cũ có cấu hình khác; dùng notebook này để xem so sánh công bằng giữa các model trong framework mới.


## 1. Nạp dữ liệu và cấu hình

Trên Kaggle, thêm copper_investing_2006_2026.csv trong phần Input. Notebook tự tìm theo tên file. Khi chạy local từ thư mục repo, notebook đọc data/copper_investing_2006_2026.csv.


In [ ]:
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

SEEDS = [42]  # Tăng độ tin cậy bằng [42, 43, 44], đổi lại sẽ train lâu hơn.
LOOKBACK = 30
TRAIN_RATIO, VALIDATION_RATIO = 0.80, 0.10
EPOCHS, BATCH_SIZE, PATIENCE = 100, 32, 10
MULTISTEP_HORIZON = 5
TARGET = 'closed_copper_price'
FEATURES = [TARGET, 'close_wti_oil', 'close_gold', 'close_silver']
MODELS = ['SimpleRNN', 'LSTM', 'GRU', 'Bi-LSTM']
CSV_NAME = 'copper_investing_2006_2026.csv'

data_path = Path('data') / CSV_NAME
kaggle_input = Path('/kaggle/input')
if kaggle_input.exists():
    candidates = sorted(kaggle_input.rglob(CSV_NAME))
    if candidates:
        data_path = candidates[0]
if not data_path.exists():
    raise FileNotFoundError(
        f'Không tìm thấy {CSV_NAME}. Hãy attach CSV trong Kaggle Input '
        'hoặc đặt file trong data/.'
    )

df = pd.read_csv(data_path)
needed = {'date', *FEATURES}
missing = sorted(needed - set(df.columns))
if missing:
    raise ValueError(f'Dataset thiếu cột: {missing}')
df['date'] = pd.to_datetime(df['date'], errors='raise')
df = df.sort_values('date', kind='stable').reset_index(drop=True)
if df['date'].isna().any() or df['date'].duplicated().any():
    raise ValueError('Ngày bị rỗng hoặc trùng.')
df[FEATURES] = df[FEATURES].apply(pd.to_numeric, errors='raise')
if not np.isfinite(df[FEATURES].to_numpy(dtype=float)).all():
    raise ValueError('Các biến đầu vào có missing hoặc giá trị không hữu hạn.')

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Dataset:', data_path, '| shape:', df.shape)
print('Date range:', df['date'].min().date(), '->', df['date'].max().date())


## 2. Hàm chuẩn bị dữ liệu và mô hình

Chia thời gian 80/10/10, chỉ fit MinMaxScaler trên train, dùng lookback 30. Bốn model dùng Adam 0.001, MSE, batch 32, tối đa 100 epoch và early stopping patience 10.


In [ ]:
def prepare_data(frame, input_columns, target_column, lookback, horizon=1):
    n = len(frame)
    train_end = int(np.floor(n * TRAIN_RATIO))
    val_end = int(np.floor(n * (TRAIN_RATIO + VALIDATION_RATIO)))
    if not (0 < train_end < val_end < n):
        raise ValueError('Không đủ dữ liệu để chia train/validation/test.')

    x_raw = frame[input_columns].to_numpy(dtype=float)
    y_raw = frame[[target_column]].to_numpy(dtype=float)
    x_scaler, y_scaler = MinMaxScaler(), MinMaxScaler()
    x_scaler.fit(x_raw[:train_end])
    y_scaler.fit(y_raw[:train_end])
    x_scaled = x_scaler.transform(x_raw)

    def make_batch(start, end):
        first_origin = max(lookback - 1, start - 1)
        last_origin = min(end - horizon - 1, n - horizon - 1)
        origins = np.arange(first_origin, last_origin + 1)
        if len(origins) == 0:
            raise ValueError(f'Không đủ cửa sổ cho horizon={horizon}.')
        X = np.stack([
            x_scaled[o - lookback + 1:o + 1] for o in origins
        ]).astype('float32')
        indices = origins[:, None] + np.arange(1, horizon + 1)
        raw_y = y_raw[indices, 0]
        y = y_scaler.transform(raw_y.reshape(-1, 1))
        y = y.reshape(-1, horizon).astype('float32')
        return {'X': X, 'y': y, 'origins': origins, 'indices': indices}

    return {
        'train': make_batch(0, train_end),
        'val': make_batch(train_end, val_end),
        'test': make_batch(val_end, n),
        'x_scaler': x_scaler,
        'y_scaler': y_scaler,
        'train_end': train_end,
        'val_end': val_end,
    }


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def build_model(name, input_shape, output_steps=1):
    inputs = tf.keras.Input(shape=input_shape)
    if name == 'SimpleRNN':
        hidden = tf.keras.layers.SimpleRNN(32)(inputs)
    elif name == 'LSTM':
        hidden = tf.keras.layers.LSTM(32)(inputs)
    elif name == 'GRU':
        hidden = tf.keras.layers.GRU(32)(inputs)
    elif name == 'Bi-LSTM':
        hidden = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(32)
        )(inputs)
    else:
        raise ValueError(f'Model không hỗ trợ: {name}')
    outputs = tf.keras.layers.Dense(output_steps)(hidden)
    model = tf.keras.Model(inputs, outputs, name=name.replace('-', '_'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
    )
    return model


def fit_model(model, train, val):
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', mode='min', patience=PATIENCE,
        restore_best_weights=True,
    )
    started = time.perf_counter()
    history = model.fit(
        train['X'], train['y'],
        validation_data=(val['X'], val['y']),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        shuffle=False, callbacks=[early_stop], verbose=0,
    )
    seconds = time.perf_counter() - started
    best_epoch = int(np.argmin(history.history['val_loss'])) + 1
    return history, seconds, best_epoch


def inverse_values(scaler, values):
    arr = np.asarray(values)
    return scaler.inverse_transform(arr.reshape(-1, 1)).reshape(arr.shape)


def metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float).reshape(-1)
    predicted = np.asarray(predicted, dtype=float).reshape(-1)
    error = predicted - actual
    mse = float(np.mean(error ** 2))
    mape = (
        float(np.mean(np.abs(error / actual)) * 100)
        if np.all(np.abs(actual) > 1e-8) else np.nan
    )
    return {
        'mse': mse,
        'rmse': float(np.sqrt(mse)),
        'mae': float(np.mean(np.abs(error))),
        'mape_pct': mape,
    }


def output_folder():
    root = Path('/kaggle/working')
    if not root.exists():
        root = Path.cwd()
    folder = root / 'all_algorithm_comparison'
    folder.mkdir(parents=True, exist_ok=True)
    return folder


results_dir = output_folder()
print('Kết quả sẽ lưu tại:', results_dir)


## 3. So sánh bốn kiến trúc dự báo một phiên

Mỗi model được train lại từ đầu trên cùng split. Metric tính trên giá gốc USD/pound. Improvement dương nghĩa là lỗi giảm so với SimpleRNN.


In [ ]:
one_step = prepare_data(df, FEATURES, TARGET, LOOKBACK, horizon=1)
rows = []
predictions_by_model = {}

for seed in SEEDS:
    for name in MODELS:
        print(f'Training {name} | seed={seed}')
        tf.keras.backend.clear_session()
        set_seed(seed)
        model = build_model(name, one_step['train']['X'].shape[1:])
        _, train_seconds, best_epoch = fit_model(
            model, one_step['train'], one_step['val']
        )
        pred_scaled = model(one_step['test']['X'], training=False).numpy()
        pred = inverse_values(one_step['y_scaler'], pred_scaled).reshape(-1)
        actual = inverse_values(
            one_step['y_scaler'], one_step['test']['y']
        ).reshape(-1)
        rows.append({
            'model': name, 'seed': seed,
            **metrics(actual, pred),
            'best_epoch': best_epoch,
            'training_seconds': train_seconds,
            'parameters': int(model.count_params()),
        })
        predictions_by_model[(name, seed)] = {
            'actual': actual,
            'prediction': pred,
            'dates': df.loc[one_step['test']['indices'][:, 0], 'date'].to_numpy(),
        }

one_step_metrics = pd.DataFrame(rows)
summary = one_step_metrics.groupby('model', as_index=False).mean(numeric_only=True)
base = summary.loc[summary['model'] == 'SimpleRNN'].iloc[0]
for metric_name in ['mse', 'rmse', 'mae']:
    summary[f'{metric_name}_improvement_pct'] = (
        (base[metric_name] - summary[metric_name]) / base[metric_name] * 100
    )
summary = summary.sort_values('rmse')
display(summary.round(5))

one_step_metrics.to_csv(results_dir / 'one_step_metrics_by_seed.csv', index=False)
summary.to_csv(results_dir / 'one_step_model_comparison.csv', index=False)

ref = predictions_by_model[(MODELS[0], SEEDS[0])]
pred_frame = pd.DataFrame({'date': ref['dates'], 'actual': ref['actual']})
for name in MODELS:
    pred_frame[name] = predictions_by_model[(name, SEEDS[0])]['prediction']
pred_frame.to_csv(results_dir / 'one_step_predictions.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(ref['dates'], ref['actual'], label='Thực tế', linewidth=2)
for name in MODELS:
    run = predictions_by_model[(name, SEEDS[0])]
    axes[0].plot(run['dates'], run['prediction'], label=name)
axes[0].set_title('Dự báo một phiên trên cùng tập test')
axes[0].set_ylabel('Giá đồng (USD/pound)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(alpha=0.25)
axes[0].legend()

ordered = summary.set_index('model').loc[MODELS]
x = np.arange(len(MODELS))
width = 0.36
axes[1].bar(x - width / 2, ordered['rmse'], width, label='RMSE')
axes[1].bar(x + width / 2, ordered['mae'], width, label='MAE')
axes[1].set_xticks(x, MODELS)
axes[1].set_title('Sai số model (thấp hơn là tốt hơn)')
axes[1].set_ylabel('USD/pound')
axes[1].grid(axis='y', alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.savefig(results_dir / 'one_step_model_comparison.png', dpi=160)
plt.show()


## 4. So sánh chiến lược GRU-IMS và GRU-DMS

IMS lặp dự báo một bước và nạp kết quả dự đoán vào cửa sổ kế tiếp. DMS xuất trực tiếp 5 giá trị trong một lần. Cả hai dùng cùng giá đồng làm input, cùng split, cùng scaler và cùng các test origin có đủ 5 target.


In [ ]:
H = MULTISTEP_HORIZON
ims_data = prepare_data(df, [TARGET], TARGET, LOOKBACK, horizon=1)
dms_data = prepare_data(df, [TARGET], TARGET, LOOKBACK, horizon=H)
if (ims_data['train_end'], ims_data['val_end']) != (
    dms_data['train_end'], dms_data['val_end']
):
    raise RuntimeError('IMS và DMS đang có split khác nhau.')

window_by_origin = {
    int(origin): window
    for origin, window in zip(
        ims_data['test']['origins'], ims_data['test']['X']
    )
}
common_origins = dms_data['test']['origins']
common_windows = np.stack([
    window_by_origin[int(origin)] for origin in common_origins
])
actual_scaled = dms_data['test']['y']
actual_multi = inverse_values(dms_data['y_scaler'], actual_scaled)

def predict_ims(model, windows, horizon):
    windows = np.asarray(windows, dtype='float32').copy()
    output = np.empty((len(windows), horizon), dtype='float32')
    for lead in range(horizon):
        step = model(windows, training=False).numpy().reshape(-1)
        output[:, lead] = step
        windows = np.concatenate(
            [windows[:, 1:, :], step[:, None, None]], axis=1
        )
    return output


multi_rows = []
multi_predictions = {}
for seed in SEEDS:
    tf.keras.backend.clear_session()
    set_seed(seed)
    ims_model = build_model(
        'GRU', ims_data['train']['X'].shape[1:], output_steps=1
    )
    _, ims_seconds, ims_best = fit_model(
        ims_model, ims_data['train'], ims_data['val']
    )
    ims_scaled = predict_ims(ims_model, common_windows, H)
    ims_pred = inverse_values(dms_data['y_scaler'], ims_scaled)

    tf.keras.backend.clear_session()
    set_seed(seed)
    dms_model = build_model(
        'GRU', dms_data['train']['X'].shape[1:], output_steps=H
    )
    _, dms_seconds, dms_best = fit_model(
        dms_model, dms_data['train'], dms_data['val']
    )
    dms_scaled = dms_model(dms_data['test']['X'], training=False).numpy()
    dms_pred = inverse_values(dms_data['y_scaler'], dms_scaled)

    multi_predictions[('GRU-IMS', seed)] = ims_pred
    multi_predictions[('GRU-DMS', seed)] = dms_pred
    for strategy, matrix, seconds, best in [
        ('GRU-IMS', ims_pred, ims_seconds, ims_best),
        ('GRU-DMS', dms_pred, dms_seconds, dms_best),
    ]:
        for lead in range(H):
            multi_rows.append({
                'strategy': strategy, 'seed': seed, 'lead': lead + 1,
                **metrics(actual_multi[:, lead], matrix[:, lead]),
                'training_seconds': seconds, 'best_epoch': best,
            })

multi_metrics = pd.DataFrame(multi_rows)
multi_summary = (
    multi_metrics.groupby(['strategy', 'lead'], as_index=False)
    .mean(numeric_only=True)
)
display(multi_summary.round(6))
multi_metrics.to_csv(results_dir / 'multistep_metrics_by_seed.csv', index=False)
multi_summary.to_csv(results_dir / 'multistep_strategy_comparison.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for strategy in ['GRU-IMS', 'GRU-DMS']:
    part = multi_summary[multi_summary['strategy'] == strategy]
    axes[0].plot(part['lead'], part['rmse'], marker='o', label=strategy)
    axes[1].plot(part['lead'], part['mae'], marker='o', label=strategy)
axes[0].set_title('RMSE theo lead')
axes[1].set_title('MAE theo lead')
for ax in axes:
    ax.set_xlabel('Lead (phiên giao dịch)')
    ax.set_ylabel('USD/pound')
    ax.set_xticks(range(1, H + 1))
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()
plt.savefig(results_dir / 'multistep_strategy_comparison.png', dpi=160)
plt.show()

position = -1
indices = dms_data['test']['indices'][position]
dates = df.loc[indices, 'date']
plt.figure(figsize=(9, 4.5))
plt.plot(dates, actual_multi[position], marker='o', label='Thực tế')
for strategy in ['GRU-IMS', 'GRU-DMS']:
    plt.plot(
        dates,
        multi_predictions[(strategy, SEEDS[0])][position],
        marker='o',
        label=strategy,
    )
plt.title('Ví dụ dự báo từ cùng một origin cuối tập test')
plt.ylabel('Giá đồng (USD/pound)')
plt.xticks(rotation=30)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(results_dir / 'multistep_example_forecast.png', dpi=160)
plt.show()
print('Các bảng và hình đã lưu tại:', results_dir)


## Cách kết luận

- Bảng kiến trúc: MSE, RMSE, MAE và MAPE thấp hơn là tốt hơn. Improvement dương nghĩa là giảm lỗi so với SimpleRNN; improvement âm nghĩa là kém hơn baseline.
- Bảng IMS/DMS: so sánh hai chiến lược tại cùng lead. Sai số thường khác nhau theo khoảng dự báo.
- Không so sánh trực tiếp metric của bảng một bước với bảng nhiều bước: bảng đầu dùng 4 biến đầu vào và dự báo 1 phiên; bảng sau chỉ dùng giá đồng và dự báo 5 phiên.
- Mặc định chạy seed 42. Có thể thêm 43 và 44 trong SEEDS để xem kết quả có ổn định không.
- Output CSV và biểu đồ được lưu trong thư mục all_algorithm_comparison.
